# Can an average conceal the important failure?

Chapter 7 · Day 10

Find a constructed rare-format regression and a deliberately leaked item. These are instrument tests, not empirical model evaluations.

Use the cycle: prediction → inspect mechanism → change one choice → interpret limits. Runnable reference answers follow each question. All computations use CPU; no model download or server is started.

In [ ]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/dongxi_llms").is_dir())
sys.path.insert(0, str(root / "src"))
import math, copy, json
import numpy as np
import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white",
                     "font.family": "DejaVu Sans", "font.size": 11})
from dongxi_llms import evaluation_lab as ev
from dongxi_llms import instruction_data_lab as data
from dongxi_llms import sft_lab as sft
def show_flow(labels, focus):
    fig, ax = plt.subplots(figsize=(11, 1.9))
    for i, label in enumerate(labels):
        ax.text(i, .5, label, ha="center", va="center",
                bbox=dict(boxstyle="round,pad=.45", fc="#dce9f5" if i == focus else "#f6f6f6", ec="#666"))
        if i: ax.annotate("", (i-.25,.5), (i-.75,.5), arrowprops=dict(arrowstyle="->",color="#555"))
    ax.set_xlim(-.6,len(labels)-.4); ax.set_ylim(0,1); ax.axis("off")
    ax.set_title("Schematic process: highlighted component is this lesson's focus")
    plt.show()


In [ ]:
show_flow(["Sources","Grouped splits","Per-item scores","Slices"], 1)

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![03 slices and contamination: reference plot 01](../figures/chapter-07/day-10-03_slices_and_contamination-01.png)

This fixed image comes from the CPU reference or the explicitly identified saved evidence; changing inputs does not update the preview automatically.

## Predict 1

A new model improves the total average. Must it improve every task family?

Pause and explain your reasoning before running the next cell.

### Reference answer and mechanism

No. An aggregate weights common tasks more heavily. A rare slice can regress while the global score improves.

In [ ]:
rows = ev.evaluation_fixture()
reports = {key: ev.slice_scores([dict(slice=r["slice"],correct=r[key]) for r in rows]) for key in ("a","b")}
names = list(reports["a"])
x = np.arange(len(names)); fig,ax = plt.subplots(figsize=(7,3.5))
for offset,key,color in [(-.18,"a","#346c9c"),(.18,"b","#b16b31")]:
    ax.bar(x+offset,[reports[key][n]["accuracy"] for n in names],width=.35,label=key.upper(),color=color)
ax.set(xticks=x,xticklabels=[n+" (n="+str(reports["a"][n]["n"])+")" for n in names],ylim=(0,1.05),ylabel="Accuracy",title="Purpose-built fixture: expose the rare slice")
ax.legend(); plt.show()
print({k:np.mean([r[k] for r in rows]) for k in ("a","b")})

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![03 slices and contamination: reference plot 02](../figures/chapter-07/day-10-03_slices_and_contamination-02.png)

This fixed image comes from the CPU reference or the explicitly identified saved evidence; changing inputs does not update the preview automatically.

## Predict 2

Will a normalized exact-content hash detect a train/test duplicate with different capitalization? Will it detect a paraphrase?

Pause and explain your reasoning before running the next cell.

### Reference answer and mechanism

It detects the constructed case/whitespace duplicate. It does not prove semantic independence; paraphrases and shared document sources need separate audits.

In [ ]:
records = [dict(id="train-1",split="train",prompt="Copy RED",answer="red"),
           dict(id="test-1",split="test",prompt="copy red",answer=" RED "),
           dict(id="test-2",split="test",prompt="Repeat the red word",answer="red")]
leaks = ev.contamination_groups(records)
assert len(leaks)==1
print("Cross-split exact groups:",leaks)

## Predict 3

Design a repair that preserves evidence about the original leakage and avoids putting related records in different splits.

Pause and explain your reasoning before running the next cell.

### Reference answer and mechanism

Keep the failed audit, group related source records before splitting, create a new contract revision, and rerun the baseline. Do not quietly delete only inconvenient errors.

In [ ]:
repaired = [{**r,"split":"train" if r["id"] in ("train-1","test-1") else r["split"]} for r in records]
assert not ev.contamination_groups(repaired)
print("New source contract hash:",ev.canonical_hash(repaired))

## Reading the evidence

The figures are regenerated from the current lesson tensors or declared fixture. Axes and counts define the comparison. Change one control, rerun the relevant cell, and explain what changed in the mechanism. Separate a verified implementation property from a claim about a real language model.

Return to the corresponding chapter and worked solution for the complete argument.